# 02 · Fetch rent and income from SCB

**Source:** Statistics Sweden (SCB), [PxWebApi 2](https://www.scb.se/en/services/open-data-api/pxwebapi/). Free, no API key.

| Table | Content used |
|---|---|
| `TAB4590` | Median annual rent per m² in rental apartments, and its margin of error. Sample survey |
| `TAB3554` | Median total earned income (sammanräknad förvärvsinkomst), ages 20–64, both sexes. Register data |

The raw responses are saved exactly as received → `data_raw/`

In [ ]:
from io import StringIO

import pandas as pd
import requests

In [ ]:
BASE_URL = "https://statistikdatabasen.scb.se/api/v2/tables"
YEARS = "2020,2021,2022,2023,2024"

# "vs_RegionKommun07EjAggr" = the 290 municipalities only, no regions or country total
MUNICIPALITIES_ONLY = "vs_RegionKommun07EjAggr"

## Helper

`params` is a dictionary of filters; `requests` turns it into the URL query string and encodes special characters (e.g. `+` → `%2B`).
The response is requested as CSV.

**Missing values:** SCB writes `..` when a value is not published (usually too small a sample).
`na_values=[".."]` reads these as missing (NaN) instead of text.
**Municipality codes** are read as text so leading zeros are kept (`0180`, not `180`).

In [ ]:
def fetch_scb(table: str, params: dict, raw_path: str) -> pd.DataFrame:
    """Download an SCB table as CSV, save the raw text, return a DataFrame."""
    response = requests.get(f"{BASE_URL}/{table}/data", params=params, timeout=60)
    response.raise_for_status()

    # save the response exactly as received (including the ".." markers)
    with open(raw_path, "w", encoding="utf-8") as f:
        f.write(response.text)

    return pd.read_csv(
        StringIO(response.text),
        dtype={"Region": str},
        na_values=[".."],
    )

## Rent

`000000J4` = median rent, `000000J3` = margin of error of the median. `Ah_kvm` = annual rent per m².

In [ ]:
rent = fetch_scb(
    "TAB4590",
    {
        "lang": "sv",
        "valueCodes[Region]": "*",
        "codelist[Region]": MUNICIPALITIES_ONLY,
        "valueCodes[Hyresuppg]": "Ah_kvm",
        "valueCodes[ContentsCode]": "000000J4,000000J3",
        "valueCodes[Tid]": YEARS,
        "outputFormat": "csv",
    },
    "../data_raw/scb_rent_raw.csv",
)

print(rent.shape)                 # 290 municipalities
print(rent.isna().sum())          # 1, 2, 1, 0, 2 missing medians for 2020–2024

## Income

`HE0110J8` = median income (thousand SEK). Ages 20–64 match the unemployment KPI.

In [ ]:
income = fetch_scb(
    "TAB3554",
    {
        "lang": "sv",
        "valueCodes[ContentsCode]": "HE0110J8",
        "valueCodes[Region]": "*",
        "codelist[Region]": MUNICIPALITIES_ONLY,
        "valueCodes[Kon]": "1+2",           # both sexes
        "valueCodes[Alder]": "20-64",
        "valueCodes[Inkomstklass]": "TOT",  # all income classes
        "valueCodes[Tid]": YEARS,
        "outputFormat": "csv",
    },
    "../data_raw/scb_income_raw.csv",
)

print(income.shape)
print(income.isna().sum())        # register data: no missing values